In [1]:
import pandas as pd
import numpy as np
import re
import sys
sys.path.append("..")

In [2]:
from src.cleaning import clean_text, clean_price, clean_amazon, clean_google

In [3]:
df1 = pd.read_csv("../data/raw/Amazon.csv", encoding="latin-1")
df2 = pd.read_csv("../data/raw/GoogleProducts.csv", encoding="latin-1")
truth = pd.read_csv("../data/raw/Amzon_GoogleProducts_perfectMapping.csv", encoding="latin-1")

In [4]:
print(df1["price"].head(10).tolist())
print(df2["price"].head(10).tolist())
print(df1["price"].dtype, df2["price"].dtype)

[0.0, 0.0, 0.0, 599.99, 99.99, 0.0, 499.99, 399.54, 19.99, 14.99]
['38.99', '8.49', '637.99', '12.95', '805.99', '103.99', '585.99', '1054.99', '25.99', '92.99']
float64 object


In [5]:
print(df1[df1["price"].astype(str).str.contains("[a-zA-Z]")]["price"].head())

print(df1[df1["title"].str.contains(r"[^\x00-\x7f]")]["title"].head())
print(df2[df2["name"].str.contains(r"[^\x00-\x7f]")]["name"].head())


print(df1["description"].str.contains("<", na=False).sum())
print(df2["description"].str.contains("<", na=False).sum())

# duplicates
print(df1.duplicated().sum(), df2.duplicated().sum())

Series([], Name: price, dtype: float64)
1113    logic express 7.2 upgrade from logic  express 6
Name: title, dtype: object
3213    adobe photoshop elements 5.0 plus adobe premie...
Name: name, dtype: object
1
6
0 0


In [6]:
print(repr(df1.loc[1113, "title"]))
print(repr(df2.loc[3213, "name"]))

print(df1[df1["description"].str.contains("<", na=False)]["description"].head(1).tolist())
print(df2[df2["description"].str.contains("<", na=False)]["description"].head(3).tolist())

print(df1["title"].duplicated().sum(), df2["name"].duplicated().sum())

'logic express 7.2 upgrade from logic \xa0express 6'
'adobe photoshop elements 5.0 plus adobe premiere elements 3.0 â?? with teacher resource center cd'
["tonka town comes to life on your pc! join chuck the dump truck and all of his tonka friends in the town where every building and vehicle talks! drive around and explore! help your friends! put out fires with hook 'n ladder!</"]
["try on the new look you've always wanted! snap a digital photo of yourself then unleash your imagination with this virtual beauty salon. experiment with hairstyles makeup eyewear jewelry and more! system requirements: for windows: <l", 'adobe® acrobat® 8 professional software enables business professionals to reliably create combine and control adobe pdf documents for easy more secure distribution collaboration and data collection. system requirements: <l', "jump into chess action move by move to rescue chesster from deep within king black's castle. join fritz and bianca in a thrilling chess adventure where 

In [7]:
dup = df2[df2["name"].duplicated(keep=False)].sort_values("name")
print(dup[["id", "name", "price"]].head(12))
dup_ids = set(df2[df2["name"].duplicated(keep=False)]["id"])
print(truth["idGoogleBase"].isin(dup_ids).sum(), "of", len(df1))

                                                     id  \
1149  http://www.google.com/base/feeds/snippets/1271...   
1147  http://www.google.com/base/feeds/snippets/5973...   
2679  http://www.google.com/base/feeds/snippets/1799...   
2887  http://www.google.com/base/feeds/snippets/6701...   
2903  http://www.google.com/base/feeds/snippets/5363...   
2701  http://www.google.com/base/feeds/snippets/1024...   
2694  http://www.google.com/base/feeds/snippets/3705...   
2905  http://www.google.com/base/feeds/snippets/5382...   
2692  http://www.google.com/base/feeds/snippets/1025...   
2885  http://www.google.com/base/feeds/snippets/2116...   
2678  http://www.google.com/base/feeds/snippets/1794...   
2912  http://www.google.com/base/feeds/snippets/1660...   

                                                   name     price  
1149                          1000pk multi pack instant  71876.95  
1147                          1000pk multi pack instant  71876.95  
2679  abacus software 155755

In [8]:
multi = truth.groupby("idAmazon").filter(lambda g: len(g) > 1)
multi = multi.merge(df2[["id", "name"]], left_on="idGoogleBase", right_on="id")

names_per_amazon = multi.groupby("idAmazon")["name"].nunique()
print("all matches same name:", (names_per_amazon == 1).sum(), "of", multi["idAmazon"].nunique())
example_ids = names_per_amazon[names_per_amazon > 1].index[:3]
for i in example_ids:
    print(df1[df1["id"] == i]["title"].values[0])
    print(multi[multi["idAmazon"] == i]["name"].tolist())
    print()

all matches same name: 11 of 144
punch super home suite 3.0
['punch software 17100 punch! super home suite', 'punch software 17100 punch! super home suite', 'punch software 17100 - punch! super home suite v.3.0 - complete product - architectural - 1 user - complete productcomplete productstandard - english - pc', 'punch software 17100 punch! super home suite']

punch 5 in 1 home design
['punch software 20100 punch! 5 in 1 home design', 'punch software 20100 - punch! home design - complete product - architectural - 1 user - complete productcomplete productstandard - english - pc', 'punch software 20100 punch! 5 in 1 home design', 'punch software 20100 punch! 5 in 1 home design']

punch! professional home design
['punch software 14100 punch! professional home design suite for windows', 'punch software 14100 punch! professional home design suite for windows', 'punch software 14100 - punch! home design suite professional - architectural - 1 user(s) - pc', 'punch software 14100 punch! profe

In [9]:
print(df1["title"].str.contains("&amp;", na=False).sum())
print(df2["name"].str.contains("&amp;", na=False).sum())
print(df2["description"].str.contains("&amp;", na=False).sum())

0
0
0


In [10]:
amazon_c = clean_amazon(df1)
google_c = clean_google(df2)

# before and after examples
for i in range(5):
    print("before:", df1["title"][i])
    print("after :", amazon_c["name_clean"][i])
    print()

for i in range(5):
    print("before:", df2["name"][i])
    print("after :", google_c["name_clean"][i])
    print()

print("amazon missing prices:", amazon_c["price_clean"].isnull().sum())
print("google missing prices:", google_c["price_clean"].isnull().sum())

before: clickart 950 000 - premier image pack (dvd-rom)
after : clickart 950 000 premier image pack dvd rom

before: ca international - arcserve lap/desktop oem 30pk
after : ca international arcserve lap desktop oem 30pk

before: noah's ark activity center (jewel case ages 3-8)
after : noahs ark activity center jewel case ages 3 8

before: peachtree by sage premium accounting for nonprofits 2007
after : peachtree by sage premium accounting for nonprofits 2007

before: singing coach unlimited
after : singing coach unlimited

before: learning quickbooks 2007
after : learning quickbooks 2007

before: superstart! fun with reading & writing!
after : superstart fun with reading and writing

before: qb pos 6.0 basic software
after : qb pos 6.0 basic software

before: math missions: the amazing arcade adventure (grades 3-5)
after : math missions the amazing arcade adventure grades 3 5

before: production prem cs3 mac upgrad
after : production prem cs3 mac upgrad

amazon missing prices: 199
goo

In [11]:
pairs = truth.merge(amazon_c, left_on="idAmazon", right_on="id")
pairs = pairs.merge(google_c, left_on="idGoogleBase", right_on="id", suffixes=("_amz", "_ggl"))

print("total pairs:", len(pairs))

same = (pairs["name_clean_amz"] == pairs["name_clean_ggl"]).sum()
print("identical names after cleaning:", same, "out of", len(pairs))
print("percentage:", round(same / len(pairs) * 100, 2))

total pairs: 1300
identical names after cleaning: 53 out of 1300
percentage: 4.08


In [12]:
dup_ids = set(df2[df2["name"].duplicated(keep=False)]["id"])
n_dup_pairs = truth["idGoogleBase"].isin(dup_ids).sum()
print(n_dup_pairs, "of", len(truth), "=", round(n_dup_pairs / len(truth) * 100, 1), "%")
print((df1["price"] == 0).sum(), "zero prices in amazon,", amazon_c["price_clean"].isnull().sum(), "missing after cleaning")

69 of 1300 = 5.3 %
199 zero prices in amazon, 199 missing after cleaning


In [13]:
amazon_c.to_csv("../data/interim/amazon_clean.csv", index=False)
google_c.to_csv("../data/interim/google_clean.csv", index=False)
check = pd.read_csv("../data/interim/amazon_clean.csv")
print(check.shape)
print(check.columns.tolist())

(1363, 9)
['id', 'title', 'description', 'manufacturer', 'price', 'name_clean', 'desc_clean', 'manu_clean', 'price_clean']


## Cleaning summary

### Problems found in the raw data
- **Prices:** Amazon prices are numbers (float64) and Google prices are text (object), so the two tables store the same field differently. No currency text was found in Google prices.
- **Zero prices:** Many Amazon prices are 0.0, including 136 of the 1,300 matched pairs. A real product does not cost 0, so these mean "price not available". Google has no zero prices.
- **Extreme price:** One Google listing has a price of 71,876.95 (a 1000-pack multi pack), which is almost certainly a data error.
- **HTML noise:** Only 1 Amazon and 6 Google descriptions contain "<". Several descriptions are cut off in the middle of a tag (for example ending in "<l"), so a normal tag-removal pattern would leave stray letters.
- **Encoding noise:** Only 1 Amazon title and 1 Google name contain non-ASCII characters: a non-breaking space in Amazon and a garbled symbol ("â??") in Google, caused by an encoding problem in the source file.
- **Spacing and apostrophes:** Some titles have double spaces, and apostrophes inside words (you've, black's) would split words if replaced with a space.
- **Duplicates:** There are no fully duplicated rows. However, 18 Amazon titles and 205 Google names repeat under different IDs. In the groups I checked, the copies have the same name and price and differ only in ID, so they are duplicate listings.
- **Missing values:** Google manufacturer is missing for 92.8% of records. Description is missing for 8.4% of Amazon and 5.9% of Google records.

### Cleaning rules applied (`src/cleaning.py`)
1. Missing text values become an empty string.
2. Text is lowercased.
3. HTML tags are removed, including tags that are cut off at the end of the text, and HTML entities such as `&amp;`.
4. `&` is replaced with "and".
5. Apostrophes are removed without adding a space (you've becomes youve).
6. Other punctuation is replaced with a space, but dots between digits are kept so that versions like 2.0 and 10.5 survive.
7. Extra whitespace (including non-breaking spaces) is collapsed.
8. Prices are converted to numbers, and a price of 0 is treated as missing.
9. New columns were added (`name_clean`, `desc_clean`, `manu_clean`, `price_clean`) and the original columns were kept unchanged.

### Decisions
- **Duplicates were kept.** Removing them could delete true links from the ground truth. Only 69 true pairs involve a duplicated Google name, which is about 4.08% of the 1,300 pairs.
- **Boilerplate words were not removed.** Phrases such as "complete product", "1 user" and "english" are left for the normalization step, because cleaning is only meant to make the text consistent.
- **Garbled characters were not repaired.** Only one record per table is affected, so the effect is too small to justify extra rules.

### Result
| | Identical names in matched pairs |
|---|---|
| Before cleaning | 44 of 1,300 (3.4%) |
| After cleaning | 53 of 1,300 (4.08%) |

Cleaning increases the exact-match rate only slightly, so exact matching is still not enough.

### Extra observations that guide the next steps
- 133 of the 144 Amazon records with several matches have Google matches with *different* names. One-to-many matching mostly comes from the same product being listed in different styles, not from duplicate rows.
- Google names tend to follow the pattern: manufacturer, part number, product name, then spec details. Amazon titles are shorter and have no part number.
- Some Google names contain words glued together with no space (for example "complete productcomplete productstandard").
- Amazon titles are much shorter than Google names, so similarity measures that ignore the extra words in the longer name are needed.

### Next steps
- **Normalization (Step 4):** extract the manufacturer from the Google name, extract version and model numbers, and reduce boilerplate words.
- **Features (Step 6):** use word-overlap measures that handle a short title against a long name, and use relative price difference with a missing-price flag.